# **Clustering**

## Objectives

* Extend the preprocessing pipeline with scaling/encoding appropriate for distance-based clustering
* Fit and evaluate a cluster model to group similar data
* Answer Hypothesis 4: *Distinct guest booking segments exist within the data. These segments exhibit meaningfully different cancellation rates suggesting cancellation risk is not uniform across the customer base*
* Answer Business Requirement 3: *TCS Hotels wants to identify distinct guest booking segments with meaningfully different cancellation profiles, in order to better understand the composition of their demand and inform targeted retention strategies*

## Inputs

* Validated dataset "outputs/datasets/cleaned/HotelBookingsValid.csv"
* Preprocessing pipeline "outputs/ml_pipeline/preprocessing/preprocessing_pipeline.pkl"

## Outputs

* Cluster preprocessing pipeline saved to "outputs/ml_pipeline/preprocessing/cluster_preprocessing_pipeline.pkl"
* Cluster modelling pipeline saved to "outputs/ml_pipeline/cluster_analysis/v1/cluster_model_pipeline.pkl"
* Conclusion to H4
* Decision about whether clusters could add any meaningful predictive power if included in the predictive model pipeline

## Additional Comments

* ⚠️ Split analysis into another notebook to avoid rerunning the modelling steps every time the kernel needs reset ⚠️


---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir

---

## Load Data

In [ ]:
import pandas as pd

df = pd.read_csv("outputs/datasets/cleaned/HotelBookingsValid.csv")
df.head(3)

In [ ]:
X = df.drop(columns="is_canceled")
y = df["is_canceled"]

print(X.shape, y.shape)

---

## Add model specific preprocessing

**Clustering Pipeline Actions**

|Feature|Cluster model actions|Experimental cluster model alternatives|
|---|---|---|
|is_canceled|exclude||
|lead_time|scaling|log/skew transformation|
|arrival_date_month|cyclical encoding||
|arrival_date_week_number|exclude|cyclical encoding|
|arrival_date_day_of_month|drop||
|stays_in_weekend_nights|scaling|binning|
|stays_in_week_nights|scaling|binning|
|adults|scaling|binning|
|children|binary|binary combined (children + babies)|
|babies|binary|binary combined (children + babies)|
|country|frequency|regional grouping, exclude|
|previous_cancellations|binary|binning|
|previous_bookings_not_canceled|binary|binning|
|booking_changes|binary|binning|
|agent|binary|frequency|
|days_in_waiting_list|binary|binning|
|adr|scaling|log/skew transformation; binning|
|required_car_parking_spaces|binary||
|total_of_special_requests|binning|scaling, binary|

In [ ]:
drop = ["arrival_date_day_of_month", "arrival_date_week_number"]
cyclical = ["arrival_date_month"]
frequency = ["country"]
binary = ["children",
          "babies",
          "previous_cancellations",
          "previous_bookings_not_canceled",
          "booking_changes",
          "agent",
          "days_in_waiting_list",
          "required_car_parking_spaces"]
binning = ["total_of_special_requests"]
scaling = ["lead_time",
           "stays_in_weekend_nights",
           "stays_in_week_nights",
           "adults",
           "adr",
           ]

In [ ]:
data = X.copy()
data.shape

* Load and test preprocessing pipeline

In [ ]:
import joblib
from utils.custom_transformers import undefined_meal

preprocessing_pipeline = joblib.load("outputs/ml_pipeline/preprocessing/preprocessing_pipeline.pkl")
pipeline_step1 = preprocessing_pipeline.fit_transform(data)
pipeline_step1.shape

* Check the steps have been completed correctly

In [ ]:
print(pipeline_step1.columns)
print(pipeline_step1["agent"].isnull().sum())
print(pipeline_step1["country"].isnull().sum())
pipeline_step1[["adr", "lead_time", "stays_in_week_nights", "stays_in_weekend_nights"]].describe()


---

In [ ]:
from feature_engine.selection import DropFeatures

drop_transformer = DropFeatures(features_to_drop=drop)
pipeline_step2 = drop_transformer.fit_transform(pipeline_step1)
pipeline_step2.shape

In [ ]:
pipeline_step2["arrival_date_month"].unique()

In [ ]:
from sklearn.preprocessing import FunctionTransformer
from utils.custom_transformers import month_name_to_number

number_transformer = FunctionTransformer(month_name_to_number)
pipeline_step3 = number_transformer.fit_transform(pipeline_step2)
pipeline_step3["arrival_date_month"].unique()

In [ ]:
from feature_engine.creation import CyclicalFeatures

cyclical_transformer = CyclicalFeatures(variables=cyclical, drop_original=True)
pipeline_step4 = cyclical_transformer.fit_transform(pipeline_step3)
pipeline_step4.columns

In [ ]:
from feature_engine.encoding import CountFrequencyEncoder

frequency_encoder = CountFrequencyEncoder(encoding_method="frequency", variables=frequency)
pipeline_step5 = frequency_encoder.fit_transform(pipeline_step4)
pipeline_step5["country"].head()

In [ ]:
binary_edges = {}

for col in binary:
    binary_edges[col] = [-float("inf"), 1, float("inf")]

binary_edges

In [ ]:
from feature_engine.discretisation import ArbitraryDiscretiser

binariser = ArbitraryDiscretiser(binning_dict=binary_edges)
pipeline_step6 = binariser.fit_transform(pipeline_step5)
pipeline_step6[binary]

In [ ]:
from feature_engine.discretisation import EqualFrequencyDiscretiser

binner = EqualFrequencyDiscretiser(variables=binning)
pipeline_step7 = binner.fit_transform(pipeline_step6)
pipeline_step7[binning].head()

In [ ]:
pipeline_step7[binning].describe()

In [ ]:
from feature_engine.wrappers import SklearnTransformerWrapper
from sklearn.preprocessing import StandardScaler

scaler = SklearnTransformerWrapper(StandardScaler(), variables=scaling)
pipeline_step8 = scaler.fit_transform(pipeline_step7)
pipeline_step8[scaling].head()

In [ ]:
from sklearn.pipeline import Pipeline

def cluster_preprocessing_pipeline():
    pipeline_base = Pipeline([
        ("Preprocessing", preprocessing_pipeline),
        ("DropFeatures", DropFeatures(features_to_drop=drop)),
        ("FunctionTransformer", FunctionTransformer(month_name_to_number)),
        ("CyclicalEncoding", CyclicalFeatures(variables=cyclical, drop_original=True)),
        ("FrequencyEncoding", CountFrequencyEncoder(encoding_method="frequency", variables=frequency)),
        ("BinaryEncoding", ArbitraryDiscretiser(binning_dict=binary_edges)),
        ("Binning", EqualFrequencyDiscretiser(variables=binning)),
        ("Scaling", SklearnTransformerWrapper(StandardScaler(), variables=scaling))
    ])
    return pipeline_base

In [ ]:
test_df = X.copy()
test_df.shape

In [ ]:
cluster_model_preprocessing_pipeline = cluster_preprocessing_pipeline()
test = cluster_model_preprocessing_pipeline.fit_transform(test_df)
test.shape

---

## Cluster Modelling Pipeline

* Build the clustering model pipeline

In [ ]:
X_clusters = X.copy()
X_transformed = cluster_model_preprocessing_pipeline.fit_transform(X_clusters)
print("Transformed shape is ", X_transformed.shape)
print("Transformed dtypes are ", X_transformed.dtypes)

* Evaluate different k values

In [ ]:
# Code adapted from scikit-learn.org https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html#sphx-glr-auto-examples-cluster-plot-kmeans-silhouette-analysis-py
# SPDX-License-Identifier: BSD-3-Clause

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score

X_kmeans = X_transformed.copy()

range_n_clusters = [2, 3, 4, 5, 6]

for n_clusters in range_n_clusters:
    
    clusterer = KMeans(n_clusters=n_clusters, random_state=10)
    cluster_labels = clusterer.fit_predict(X_kmeans)

    # The silhouette_score gives the average value for all the samples.
    # This gives a perspective into the density and separation of the formed
    # clusters
    silhouette_avg = silhouette_score(X_kmeans, cluster_labels, sample_size=10000, random_state=10)
    print(
        "For n_clusters =",
        n_clusters,
        "The average silhouette_score is :",
        silhouette_avg,
    )


* Remove room type columns to explore if high cardinalty one-hot encoded features is diminishing the results since room type is unlikely to be a main factor in guest behaviour patterns and holds the highest cardinality

In [ ]:
X_transformed.columns

In [ ]:
drop_cols = ['reserved_room_type_C',
       'reserved_room_type_A', 'reserved_room_type_D', 'reserved_room_type_E',
       'reserved_room_type_G', 'reserved_room_type_F', 'reserved_room_type_H',
       'reserved_room_type_L', 'assigned_room_type_C', 'assigned_room_type_A',
       'assigned_room_type_D', 'assigned_room_type_E', 'assigned_room_type_G',
       'assigned_room_type_F', 'assigned_room_type_I', 'assigned_room_type_B',
       'assigned_room_type_H', 'assigned_room_type_L']

X_kmeans = X_transformed.copy().drop(labels=drop_cols, axis=1)

range_n_clusters = [2, 3, 4, 5, 6]

for n_clusters in range_n_clusters:
    
    clusterer = KMeans(n_clusters=n_clusters, random_state=10)
    cluster_labels = clusterer.fit_predict(X_kmeans)

    # The silhouette_score gives the average value for all the samples.
    # This gives a perspective into the density and separation of the formed
    # clusters
    silhouette_avg = silhouette_score(X_kmeans, cluster_labels, sample_size=10000, random_state=10)
    print(
        "For n_clusters =",
        n_clusters,
        "The average silhouette_score is :",
        silhouette_avg,
    )

* There is insufficient improvement to justify the removal of the room type columns moving forward

* Apply PCA to the data

In [ ]:
from sklearn.decomposition import PCA

pca_test = X_transformed.copy()
pca = PCA(n_components=50, random_state=10)
pca_result = pca.fit_transform(pca_test)

print(pca_result.shape, '\n', type(pca_result))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

n_components = 20


def pca_components_analysis(df_pca, n_components):
    pca = PCA(n_components=n_components).fit(df_pca)
    x_PCA = pca.transform(df_pca)  # array with transformed PCA

    ComponentsList = ["Component " + str(number)
                      for number in range(n_components)]
    dfExplVarRatio = pd.DataFrame(
        data=np.round(100 * pca.explained_variance_ratio_, 3),
        index=ComponentsList,
        columns=['Explained Variance Ratio (%)'])

    dfExplVarRatio['Accumulated Variance'] = dfExplVarRatio['Explained Variance Ratio (%)'].cumsum(
    )

    PercentageOfDataExplained = dfExplVarRatio['Explained Variance Ratio (%)'].sum(
    )

    print(
        f"* The {n_components} components explain {round(PercentageOfDataExplained,2)}% of the data \n")
    plt.figure(figsize=(9, 6))
    sns.lineplot(data=dfExplVarRatio,  marker="o")
    plt.xticks(rotation=90)
    plt.yticks(np.arange(0, 110, 10))
    plt.show()


pca_components_analysis(df_pca=X_transformed.copy(), n_components=n_components)

In [ ]:
pca_components_analysis(df_pca=X_transformed.copy(), n_components=9)

In [ ]:
df = X_transformed.copy()
pca = PCA(n_components=9, random_state=10)
pca_df = pca.fit_transform(df)

print(pca_df.shape, '\n', type(pca_df))

In [ ]:
X_silhouette = pca_df.copy()

range_n_clusters = [2, 3, 4, 5, 6]

for n_clusters in range_n_clusters:
    
    clusterer = KMeans(n_clusters=n_clusters, random_state=10)
    cluster_labels = clusterer.fit_predict(X_silhouette)

    # The silhouette_score gives the average value for all the samples.
    # This gives a perspective into the density and separation of the formed
    # clusters
    silhouette_avg = silhouette_score(X_silhouette, cluster_labels, sample_size=10000, random_state=10)
    print(
        "For n_clusters =",
        n_clusters,
        "The average silhouette_score is :",
        silhouette_avg,
    )

In [ ]:
ks = range(1, 11)
inertias = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=10)
    km.fit(pca_df)
    inertias.append(km.inertia_)

plt.figure(figsize=(6,4))
plt.plot(ks, inertias, marker='o')
plt.xticks(ks)
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (WCSS)")
plt.title("Elbow Method")
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

scores = []

for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=10)
    labels = km.fit_predict(pca_df)
    scores.append(silhouette_score(pca_df, labels, sample_size=10000, random_state=0))

plt.plot(range(2, 11), scores, marker='o')
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.show()

In [ ]:
sample_size = 5000
random_state = 10

for n_clusters in [4, 5]:
    fig, ax1 = plt.subplots(figsize=(8, 6))

    clusterer = KMeans(n_clusters=n_clusters, random_state=random_state)
    cluster_labels = clusterer.fit_predict(pca_df)

    # Sample rows and their corresponding labels
    rng = np.random.RandomState(random_state)
    sample_idx = rng.choice(len(pca_df), size=sample_size, replace=False)

    X_sample = pca_df[sample_idx]          
    labels_sample = cluster_labels[sample_idx]

    silhouette_avg = silhouette_score(X_sample, labels_sample)
    print("For n_clusters =", n_clusters, "The average silhouette_score is :", silhouette_avg)

    sample_silhouette_values = silhouette_samples(X_sample, labels_sample)

    ax1.set_ylim([0, len(X_sample) + (n_clusters + 1) * 10])

    y_lower = 10
    for i in range(n_clusters):
        ith_cluster_silhouette_values = sample_silhouette_values[labels_sample == i]
        ith_cluster_silhouette_values.sort()

        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i

        color = plt.colormaps["winter"](float(i) / n_clusters)
        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_silhouette_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10

    ax1.set_title(f"Silhouette plot for n_clusters = {n_clusters} (sampled)")
    ax1.set_xlabel("The silhouette coefficient values")
    ax1.set_ylabel("Cluster label")
    ax1.axvline(x=silhouette_avg, color="red", linestyle="--")
    ax1.set_yticks([])
    ax1.set_xticks([-0.1, 0, 0.2, 0.4, 0.6, 0.8, 1])

plt.show()

* The model is producing a weak cluster structure. Clusters exist but have soft, overlapping boundaries

* Build the cluster pipeline with 5 clusters

In [ ]:
def cluster_pipeline():
    pipeline_base = Pipeline([
        ("Preprocessing", cluster_model_preprocessing_pipeline),
        ("PCA", PCA(n_components=9, random_state=10)),
        ("model", KMeans(n_clusters=5, random_state=10))
    ])

    return pipeline_base

* Fit and transform on the full, valid dataset

In [ ]:
print(X.shape)
X.head(3)

In [ ]:
cluster_model_pipeline = cluster_pipeline()
cluster_model_pipeline.fit(X)

In [ ]:
X["Clusters"] = cluster_model_pipeline["model"].labels_
print(X.shape)
X.head()

---

# Push files to Repo

In [ ]:
import os
try:
  os.makedirs(name='outputs/ml_pipeline/cluster_analysis/v1')
except Exception as e:
  print(e)


* Save cluster preprocessing pipeline

In [ ]:
joblib.dump(value=cluster_model_preprocessing_pipeline, filename="outputs/ml_pipeline/preprocessing/cluster_preprocessing_pipeline.pkl")

* Save Cluster modelling pipeline

In [ ]:
joblib.dump(value=cluster_model_pipeline, filename="outputs/ml_pipeline/cluster_analysis/v1/cluster_model_pipeline.pkl")